In [1]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer

import kagglehub

2026-05-03 16:13:00.524103: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 16:13:00.545263: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-03 16:13:00.545281: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-03 16:13:00.545854: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-03 16:13:00.549672: I tensorflow/core/platform/cpu_feature_guar

## 1. Data

In [2]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-2")

def load_file(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.read()

train_text = load_file(wikitext2_path, "wiki.train.tokens")
val_text   = load_file(wikitext2_path, "wiki.valid.tokens")
test_text  = load_file(wikitext2_path, "wiki.test.tokens")

## 2. Data Cleaning

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"=+.*?=+", "", text)            # poista wiki-otsikot
    text = re.sub(r"[^a-zA-Z0-9\s\.,']", "", text) # sailyta perusvalimerkit
    text = re.sub(r"\s+", " ", text)
    return text

train_text = clean_text(train_text)[:500_000]
val_text   = clean_text(val_text)[:100_000]
test_text  = clean_text(test_text)[:100_000]

## 3. Tokenizer

In [4]:
vocab_size_limit = 15000

tokenizer = Tokenizer(num_words=vocab_size_limit, oov_token="<OOV>", filters="")
tokenizer.fit_on_texts([train_text])

train_sequence = tokenizer.texts_to_sequences([train_text])[0]
val_sequence   = tokenizer.texts_to_sequences([val_text])[0]
test_sequence  = tokenizer.texts_to_sequences([test_text])[0]

word_index = tokenizer.word_index
vocab_size = min(vocab_size_limit, len(word_index) + 1)

index_to_word = {i: w for w, i in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[word_index["<OOV>"]] = "<OOV>"

seq_length = 30

## 4. Dataset

In [5]:
def build_xy(sequence, seq_len, stride=2):
    X, y = [], []
    for i in range(0, len(sequence) - seq_len, stride):
        X.append(sequence[i:i + seq_len])
        y.append(sequence[i + seq_len])

    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X_train, y_train = build_xy(train_sequence, seq_length, stride=2)
X_val, y_val     = build_xy(val_sequence, seq_length, stride=2)
X_test, y_test   = build_xy(test_sequence, seq_length, stride=2)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (46735, 30) (46735,)
Val: (9623, 30) (9623,)
Test: (9778, 30) (9778,)


## 5. Callbacks

In [20]:
# Treeniasetukset: early stopping on paalla, mutta vasta myohemmin,
# jotta malli ei katkea liian aikaisin.
USE_EARLY_STOPPING = True


def make_callbacks(ckpt_path, use_early_stopping=USE_EARLY_STOPPING):
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            save_best_only=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    if use_early_stopping:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                min_delta=0.001,
                start_from_epoch=6,
                restore_best_weights=True,
                verbose=1,
            )
        )

    return callbacks

## 6. LSTM Baseline

In [25]:
lstm_model = models.Sequential([
    layers.Embedding(vocab_size, 128),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2)),
    layers.Bidirectional(layers.LSTM(128, dropout=0.2)),
    layers.Dense(128, activation="gelu"),
    layers.Dropout(0.3),
    layers.Dense(vocab_size, activation="softmax"),
])

lstm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    metrics=["accuracy"],
)

history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=16,
    batch_size=32,
    callbacks=make_callbacks("best_lstm.keras"),
)


Epoch 1/16
1461/1461 [==============================] - 28s 18ms/step - loss: 6.8848 - accuracy: 0.0681 - val_loss: 6.7392 - val_accuracy: 0.0712 - lr: 3.0000e-04
Epoch 2/16
1461/1461 [==============================] - 21s 14ms/step - loss: 6.4659 - accuracy: 0.0961 - val_loss: 6.7246 - val_accuracy: 0.1102 - lr: 3.0000e-04
Epoch 3/16
1461/1461 [==============================] - 24s 17ms/step - loss: 6.2779 - accuracy: 0.1164 - val_loss: 6.7988 - val_accuracy: 0.1212 - lr: 3.0000e-04
Epoch 4/16
1461/1461 [==============================] - ETA: 0s - loss: 6.1586 - accuracy: 0.1214
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
1461/1461 [==============================] - 23s 16ms/step - loss: 6.1586 - accuracy: 0.1214 - val_loss: 6.9171 - val_accuracy: 0.1327 - lr: 3.0000e-04
Epoch 5/16
1461/1461 [==============================] - 24s 17ms/step - loss: 6.0315 - accuracy: 0.1252 - val_loss: 7.0080 - val_accuracy: 0.1318 - lr: 1.5000e-04
Epoch 6/16
1461/1461 [

## 7. Sampling utility

In [8]:
def sample_probs(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    preds = preds - np.max(preds)
    exp_preds = np.exp(preds)
    return exp_preds / np.sum(exp_preds)

## 8. Transformer Model

In [26]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=0.2,
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dropout(0.2),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout = layers.Dropout(0.2)

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]
        causal_mask = tf.cast(
            tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0),
            dtype=tf.bool,
        )

        attn = self.att(inputs, inputs, attention_mask=causal_mask, training=training)
        attn = self.dropout(attn, training=training)
        x = self.norm1(inputs + attn)

        ffn = self.ffn(x, training=training)
        ffn = self.dropout(ffn, training=training)
        return self.norm2(x + ffn)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.dropout = layers.Dropout(0.2)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]
        positions = tf.range(seq_len)
        return self.dropout(self.token_emb(x) + self.pos_emb(positions), training=training)


embed_dim = 128
num_heads = 4
ff_dim = 256
num_blocks = 2

inputs = layers.Input(shape=(seq_length,))
x = TokenAndPositionEmbedding(seq_length, vocab_size, embed_dim)(inputs)

for _ in range(num_blocks):
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

x = x[:, -1, :]
x = layers.Dense(128, activation="gelu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

transformer_model = keras.Model(inputs, outputs)
transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=3e-4, clipnorm=1.0),
    metrics=["accuracy"],
)


## 9. Train Transformer

In [27]:
history_t = transformer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=16,
    batch_size=32,
    callbacks=make_callbacks("best_transformer.keras"),
)

Epoch 1/16
1461/1461 [==============================] - 29s 18ms/step - loss: 6.7912 - accuracy: 0.0972 - val_loss: 6.5705 - val_accuracy: 0.1324 - lr: 3.0000e-04
Epoch 2/16
1461/1461 [==============================] - 24s 17ms/step - loss: 6.1744 - accuracy: 0.1249 - val_loss: 6.6942 - val_accuracy: 0.1478 - lr: 3.0000e-04
Epoch 3/16
1461/1461 [==============================] - ETA: 0s - loss: 5.8247 - accuracy: 0.1513
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
1461/1461 [==============================] - 23s 16ms/step - loss: 5.8247 - accuracy: 0.1513 - val_loss: 6.7632 - val_accuracy: 0.1519 - lr: 3.0000e-04
Epoch 4/16
1461/1461 [==============================] - 23s 16ms/step - loss: 5.4840 - accuracy: 0.1770 - val_loss: 6.9830 - val_accuracy: 0.1514 - lr: 1.5000e-04
Epoch 5/16
1461/1461 [==============================] - ETA: 0s - loss: 5.2915 - accuracy: 0.1925
Epoch 5: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
1461/1461 [

## 10. Top-N seuraavan sanan ennustaminen

In [11]:
def predict_top_n(text, model, top_n=5, temperature=0.8):
    """
    Ennustaa top_n todennäköisintä seuraavaa sanaa annetulle tekstille.
    Palauttaa listan (sana, prosentti) -pareista.
    """
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = seq[-seq_length:]

    if len(seq) < seq_length:
        seq = [0] * (seq_length - len(seq)) + seq

    preds = model.predict(np.array([seq]), verbose=0)[0]
    probs = sample_probs(preds, temperature)

    top_indices = np.argsort(probs)[::-1][:top_n]
    return [(index_to_word.get(i, "<UNK>"), round(float(probs[i]) * 100, 2))
            for i in top_indices]


def generate_text(seed_text, model, num_words=20, temperature=0.8):
    """Generoi tekstiä sanomalla joka askeleella todennäköisin jatko."""
    text  = seed_text
    words = text.split()

    # Kaikki tokenit joita ei haluta generoituun tekstiin
    BAD = {"<OOV>", "<PAD>", "<UNK>", "unk"}

    def detect_cycle(words, max_cycle=6):
        """Palauttaa True jos viimeiset sanat toistavat sykliä."""
        n = len(words)
        for cycle_len in range(2, max_cycle + 1):
            if n >= cycle_len * 2:
                if words[-cycle_len:] == words[-cycle_len*2:-cycle_len]:
                    return True
        return False

    for _ in range(num_words):
        candidates = predict_top_n(text, model, top_n=30, temperature=temperature)

        # Jos silmukka havaitaan, nostetaan lämpötilaa murretaksemme sen
        temp_boost = 1.5 if detect_cycle(words) else 1.0
        if temp_boost > 1.0:
            candidates = predict_top_n(text, model, top_n=30, temperature=temperature * temp_boost)

        next_word = None
        recent = set(words[-6:])  # isompi ikkuna toiston estoon

        for word, _ in candidates:
            if word in BAD:
                continue
            if word in recent:
                continue
            next_word = word
            break

        # Hätävaranto: hyväksy toistuva sana mutta ei BAD-tokeneita
        if next_word is None:
            for word, _ in candidates:
                if word not in BAD:
                    next_word = word
                    break

        if next_word is None:
            break

        words.append(next_word)
        text += " " + next_word

    return text

## 11. Esimerkkejä

In [31]:
seed = "the meaning of life"

print("=== LSTM: Top-5 seuraavaa sanaa ===")
for word, pct in predict_top_n(seed, lstm_model, top_n=5):
    print(f"  {word:20s} {pct:.2f}%")

print("\n=== Transformer: Top-5 seuraavaa sanaa ===")
for word, pct in predict_top_n(seed, transformer_model, top_n=5):
    print(f"  {word:20s} {pct:.2f}%")

print("\n=== LSTM generoitu teksti ===")
print(generate_text(seed, lstm_model, num_words=20))

print("\n=== Transformer generoitu teksti ===")
print(generate_text(seed, transformer_model, num_words=20))


=== LSTM: Top-5 seuraavaa sanaa ===
  ,                    17.26%
  .                    13.71%
  of                   7.02%
  to                   6.48%
  and                  5.93%

=== Transformer: Top-5 seuraavaa sanaa ===
  ,                    50.33%
  and                  20.74%
  in                   8.68%
  .                    8.44%
  as                   5.40%

=== LSTM generoitu teksti ===
the meaning of life , and a first gods of the new time , and a first gods of the new time , and

=== Transformer generoitu teksti ===
the meaning of life , and was not have been also is a new york . in the gods were not have been also


## 12. Validation + Test arviointi (Perplexity ja Top-5)
Lasketaan kummallekin mallille samat mittarit: loss, accuracy, perplexity ja top-5-accuracy sekä validation- että test-joukoilla.

In [32]:
def evaluate_model_metrics(model, X_eval, y_eval, batch_size=256):
    loss, acc = model.evaluate(X_eval, y_eval, batch_size=batch_size, verbose=0)

    probs = model.predict(X_eval, batch_size=batch_size, verbose=0)

    true_probs = np.clip(probs[np.arange(len(y_eval)), y_eval], 1e-9, 1.0)
    perplexity = float(np.exp(-np.mean(np.log(true_probs))))

    top5_idx = np.argpartition(probs, -5, axis=1)[:, -5:]
    top5_acc = float(np.mean((top5_idx == y_eval[:, None]).any(axis=1)))

    return {
        "loss": float(loss),
        "accuracy": float(acc),
        "perplexity": perplexity,
        "top5_accuracy": top5_acc,
    }


best_lstm_model = keras.models.load_model("best_lstm.keras")
best_transformer_model = keras.models.load_model(
    "best_transformer.keras",
    custom_objects={
        "TransformerBlock": TransformerBlock,
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
    },
)

results = []
for model_name, model_obj in [
    ("LSTM", best_lstm_model),
    ("Transformer", best_transformer_model),
]:
    for split_name, X_split, y_split in [
        ("validation", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        metrics = evaluate_model_metrics(model_obj, X_split, y_split)
        results.append({
            "model": model_name,
            "split": split_name,
            "loss": metrics["loss"],
            "accuracy": metrics["accuracy"],
            "perplexity": metrics["perplexity"],
            "top5_accuracy": metrics["top5_accuracy"],
        })

comparison_df = pd.DataFrame(results)
comparison_df[["loss", "accuracy", "perplexity", "top5_accuracy"]] = comparison_df[
    ["loss", "accuracy", "perplexity", "top5_accuracy"]
].round(4)

comparison_df

,model,split,loss,accuracy,perplexity,top5_accuracy
0,LSTM,validation,6.7246,0.1102,832.6240,0.2877
1,LSTM,test,6.8879,0.0993,980.3152,0.2544
2,Transformer,validation,6.5705,0.1324,713.7439,0.2995
3,Transformer,test,6.7187,0.1095,827.7343,0.2739


In [33]:
print("LSTM epochs run:", len(history_lstm.history["loss"]))
print("Transformer epochs run:", len(history_t.history["loss"]))

print("Best LSTM val_loss:", min(history_lstm.history["val_loss"]))
print("Best Transformer val_loss:", min(history_t.history["val_loss"]))

LSTM epochs run: 12
Transformer epochs run: 12
Best LSTM val_loss: 6.724583625793457
Best Transformer val_loss: 6.570526599884033
